<a href="https://colab.research.google.com/github/kolijayesh370-ops/walmart-supply-chain-engine/blob/main/MLmodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Load the clean dataset generated from the EDA phase
df = pd.read_csv("Walmart_Sales_Cleaned.csv")
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106


In [2]:
import pandas as pd

# Extract Week_of_Year and Year from the 'Date' column
df['Date'] = pd.to_datetime(df['Date'])
df['Week_of_Year'] = df['Date'].dt.isocalendar().week.astype(int)
df['Year'] = df['Date'].dt.year

# Selecting relevant core columns for retail forecasting and creating a copy to avoid SettingWithCopyWarning
df = df[[
    "Store",
    "Holiday_Flag",
    "Temperature",
    "Fuel_Price",
    "CPI",
    "Unemployment",
    "Week_of_Year",
    "Year",
    "Weekly_Sales"
]].copy()

# Checking and dropping any residual null values
print("Null values check per feature column:")
print(df.isnull().sum())
df.dropna(inplace=True)

Null values check per feature column:
Store           0
Holiday_Flag    0
Temperature     0
Fuel_Price      0
CPI             0
Unemployment    0
Week_of_Year    0
Year            0
Weekly_Sales    0
dtype: int64


In [3]:
# Ensure categorical IDs are treated uniformly
df["Store"] = df["Store"].astype(str)

# Removing Price Outliers using the 99th percentile threshold to eliminate extreme spikes
q99 = df["Weekly_Sales"].quantile(0.99)
df = df[df["Weekly_Sales"] <= q99]

print(f"99th percentile cutoff threshold: ${q99:,.2f}")
print(f"Remaining dataset record count: {df.shape[0]}")

99th percentile cutoff threshold: $2,404,035.30
Remaining dataset record count: 6370


In [4]:
# Encoding high-cardinality Store features into numeric format using dummy encoding
df = pd.get_dummies(df, columns=["Store"], drop_first=True)

In [5]:
# X represents the input supply chain features used for prediction
X = df.drop("Weekly_Sales", axis=1)

# y represents the target variable which is the predicted Weekly Sales dollar value
y = df["Weekly_Sales"]

In [6]:
'''
Splitting the Dataset
The dataset is divided into:
- 80% Training Data (to teach our algorithms retail patterns)
- 20% Testing Data (to validate performance metrics)
This allows the model to be evaluated on completely unseen data.
'''
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(f"Training features dimensions: {X_train.shape}")
print(f"Testing features dimensions: {X_test.shape}")

Training features dimensions: (5096, 51)
Testing features dimensions: (1274, 51)


In [7]:
'''
Linear Regression Model
This model serves as our foundational baseline for comparison.
'''
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [8]:
'''
Decision Tree Regressor
The Decision Tree learns complex decision splits directly from our supply chain properties,
predicting store-level revenue targets based on environmental conditions.
'''
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(max_depth=12, random_state=42)
dt.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=12, random_state=42)

In [9]:
'''
Random Forest Regressor
Random Forest combines multiple independent decision trees to improve prediction accuracy,
smooth out variance, and prevent model overfitting.
This model is generally highly accurate for non-linear retail sales data.
'''
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

RandomForestRegressor(n_jobs=-1, random_state=42)

In [10]:
'''
Model Evaluation
The performance of our best ensemble model is evaluated using:
- R² Score (Coefficient of Determination)
- Mean Absolute Error (MAE)
These metrics determine how accurately the model forecasts stock replenishment targets.
'''
from sklearn.metrics import r2_score, mean_absolute_error

# Generate predictions on unseen evaluation matrices
pred = rf.predict(X_test)

print("--- Final Best Model Performance (Random Forest) ---")
print(f"R² Score: {r2_score(y_test, pred):.4f}")
print(f"MAE: ${mean_absolute_error(y_test, pred):,.2f}")

--- Final Best Model Performance (Random Forest) ---
R² Score: 0.9753
MAE: $52,765.48


In [11]:
'''
Saving the Trained Model
The final trained model and feature columns are serialized using joblib.
These files will be downloaded and loaded directly into the Streamlit web dashboard application.
'''
import joblib

joblib.dump(rf, "walmart_model.pkl")
joblib.dump(X.columns.tolist(), "model_columns.pkl")

print("Successfully exported 'walmart_model.pkl' and 'model_columns.pkl' to your file space!")

Successfully exported 'walmart_model.pkl' and 'model_columns.pkl' to your file space!
